# 03 Promotions

Lesson 7: how promos are scheduled, what they cost and how many extra cups they need to pay back.

Data: `data/processed/orders_enriched.parquet` (notebook 02). STANDARD = RC000. Money is before waste.

## 0. Is the data the same as in notebook 02?

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # lets the notebook import src/

# Same chart setup as notebook 02.
pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates["fb"] = go.layout.Template(layout=dict(margin=dict(t=118, l=70, r=30, b=60), font_size=13))
pio.templates.default = "plotly_white+fb"
OUT = ROOT / "outputs"

ITEMS = ["Watermelon", "Pineapple", "Guava", "PassionFruit", "MixedBerryPremium"]
ITEM_COLORS = dict(zip(ITEMS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]))
STANDARD = "RC000"
PROMOS = ["RC101", "RC102", "RC103"]
N_MONTHS = 12


def subtitle(title, sub):
    """Chart title: the finding (split at ';' to fit), then a grey line saying what is plotted."""
    sub_html = f"<span style='font-size:12px;color:#52514e'>{sub}</span>"
    return dict(text=title.replace("; ", ";<br>") + "<br>" + sub_html, font_size=16,
                x=0.01, xanchor="left", y=1, yref="container", yanchor="top", pad=dict(t=36))


e = pd.read_parquet(ROOT / "data" / "processed" / "orders_enriched.parquet")
e["month"] = e["date"].dt.strftime("%Y-%m")
# CM if the same cups had been sold at list price: list price x (1 - commission) - fruit - pack+labor.
e["cm_list_thb"] = (e["list_price_thb"] * (1 - e["commission_rate"])
                    - e["fruit_cost_per_cup_thb"] - e["pack_labor_per_cup_thb"]) * e["units_sold"]
promo = e[e["base_code"] != STANDARD]
codes = pd.read_excel(ROOT / "data" / "raw" / "FruitBlend24_Intern_Case_Data.xlsx", sheet_name="rate_code_dim")

TOTAL_CUPS = e["units_sold"].sum()
assert len(e) == 123_655 and e["cm_thb"].notna().all()
assert set(e["base_code"]) == {STANDARD, *PROMOS}
print(f"{len(e):,} rows | {TOTAL_CUPS:,} cups | revenue {e['gross_revenue_thb'].sum():,.2f} THB | CM {e['cm_thb'].sum():,.0f} THB")
print(f"Promo rows: {len(promo):,} | promo cups: {promo['units_sold'].sum():,} ({promo['units_sold'].sum() / TOTAL_CUPS:.1%})")
display(codes[["base_code", "rate_code_name", "discount_pct", "conditions"]])

123,655 rows | 618,123 cups | revenue 31,146,932.57 THB | CM 8,496,620 THB
Promo rows: 10,647 | promo cups: 68,253 (11.0%)


,base_code,rate_code_name,discount_pct,conditions
0,RC000,Standard,0.00,No promotion. Regular menu price.
1,RC101,Bundle Deal,0.15,"Randomly scheduled single-SKU bundle discount,..."
2,RC102,Double Day New User,0.20,New-user acquisition campaign run company-wide...
3,RC103,Seasonal Push,0.12,Company-wide rainy-season (Jun-Oct) demand-boo...


123,655 rows, 618,123 cups, 31.15M THB revenue, same as notebook 02. Promos are 11% of cups: 68,253 on 3 codes.

## 1. How are promos scheduled?

In [2]:
days = (promo.groupby(["base_code", "date"])
        .agg(kitchens=("kitchen", "nunique"), platforms=("platform_code", "nunique"),
             items=("sku", "nunique"), hours=("hour", "nunique")))
schedule = days.groupby("base_code").agg(
    promo_days=("kitchens", "size"), kitchens_min=("kitchens", "min"), kitchens_max=("kitchens", "max"),
    platforms_min=("platforms", "min"), platforms_max=("platforms", "max"),
    items_min=("items", "min"), items_max=("items", "max"), hours_min=("hours", "min"), hours_max=("hours", "max"))
schedule["item_days"] = promo.groupby("base_code")[["date", "sku"]].apply(lambda d: len(d.drop_duplicates()))
display(schedule)

# RC101: on its day, does the item sell only at the bundle price, everywhere?
rc101_days = promo.loc[promo["base_code"] == "RC101", ["date", "sku"]].drop_duplicates()
on_day = e.merge(rc101_days, on=["date", "sku"])
share = on_day.groupby(["date", "sku"])["base_code"].apply(lambda c: (c == "RC101").mean())
reach = on_day.groupby(["date", "sku"]).agg(kitchens=("kitchen", "nunique"), platforms=("platform_code", "nunique"))
print(f"RC101 item-days: {len(rc101_days)} | share of the item's rows at RC101: {share.min():.0%} to {share.max():.0%}")
print(f"RC101 item-days in all 4 kitchens: {(reach['kitchens'] == 4).sum()} | on both platforms: {(reach['platforms'] == 2).sum()}")

# Hours where one item has promo and standard rows together.
flag = e.assign(is_promo=e["base_code"] != STANDARD)
mixed_kitchen = (flag.groupby(["date", "hour", "kitchen", "sku"])["is_promo"].nunique() > 1).sum()
mixed_all = (flag.groupby(["date", "hour", "sku"])["is_promo"].nunique() > 1).sum()
overlap = promo.groupby("date")["base_code"].unique()
overlap = overlap[overlap.str.len() > 1]
print(f"Item-hours with promo and RC000 rows: {mixed_kitchen} per kitchen, {mixed_all} across kitchens")
print("Days with 2 promo codes:", {d.date(): list(c) for d, c in overlap.items()})

,promo_days,kitchens_min,kitchens_max,platforms_min,platforms_max,items_min,items_max,hours_min,hours_max,item_days
base_code,,,,,,,,,,
RC101,32,4,4,1,2,1,1,18,24,32
RC102,12,4,4,2,2,4,5,24,24,54
RC103,13,4,4,2,2,4,5,24,24,59


RC101 item-days: 32 | share of the item's rows at RC101: 100% to 100%
RC101 item-days in all 4 kitchens: 32 | on both platforms: 21
Item-hours with promo and RC000 rows: 0 per kitchen, 0 across kitchens
Days with 2 promo codes: {datetime.date(2026, 6, 17): ['RC103', 'RC101']}


In [3]:
cal = promo.groupby(["base_code", "date", "sku"], as_index=False)["units_sold"].sum()
cal["row"] = np.where(cal["base_code"] == "RC101", "RC101 " + cal["sku"], cal["base_code"] + " (all items)")
rows = [f"RC101 {s}" for s in ITEMS] + ["RC102 (all items)", "RC103 (all items)"]
cal = cal.drop_duplicates(["row", "date"])
color = cal["sku"].where(cal["base_code"] == "RC101", "All items")
fig = px.scatter(cal, x="date", y="row", color=color, category_orders={"row": rows},
                 color_discrete_map={**ITEM_COLORS, "All items": "#52514e"},
                 labels={"date": "", "row": "", "color": ""})
fig.update_traces(marker=dict(size=11, symbol="square"), hovertemplate="%{y}: %{x|%a %d %b %Y}<extra></extra>")
for x0, x1 in [("2025-09-01", "2025-10-31"), ("2026-06-01", "2026-08-31")]:
    fig.add_vrect(x0=x0, x1=x1, fillcolor="#86b6ef", opacity=0.15, line_width=0)
n101 = len(rc101_days)
fig.update_layout(height=420, showlegend=False, title=subtitle(
    f"RC101 runs on {n101} item-days (3-8 per item); RC102 on all {schedule.loc['RC102', 'promo_days']} double dates; "
    f"RC103 on {schedule.loc['RC103', 'promo_days']} rainy-season days",
    "One mark per promo day. Blue bands = rainy season (Jun-Oct)."))
fig.show()

**Every promo is a whole day in all 4 kitchens: RC101 for one item (32 item-days, 3-8 per item), RC102 and RC103 for every item on sale.** On an RC101 day 100% of that item's rows are RC101; 0 item-hours mix promo and RC000.

RC102 hits all 12 double dates (1/1 ... 12/12). RC103 runs on 13 days, all in Sep-Oct 2025 and Jun-Aug 2026; on 17 Jun it shares the day with RC101 (that item takes RC101). 11 of 32 RC101 days show 1 platform: the item sold on one platform only that day (same split seen in notebook 02, 2.5).

## 2. What does each promo cost?

In [4]:
SUMS = ["units_sold", "list_revenue_thb", "discount_thb", "gross_revenue_thb", "cm_thb", "cm_list_thb"]


def promo_cost(df, by):
    """Cups, discount and CM per cup at promo vs list price. Money summed first, then divided."""
    g = df.groupby(by)[SUMS].sum()
    out = pd.DataFrame({
        "cups": g["units_sold"],
        "pct_of_all_cups": g["units_sold"] / TOTAL_CUPS * 100,
        "discount_thb_year": g["discount_thb"],
        "discount_thb_per_month": g["discount_thb"] / N_MONTHS,
        "discount_pct": g["discount_thb"] / g["list_revenue_thb"] * 100,
        "cm_per_cup_full": g["cm_list_thb"] / g["units_sold"],
        "cm_per_cup_promo": g["cm_thb"] / g["units_sold"],
        "cm_thb_promo": g["cm_thb"],
        "cm_thb_if_full_price": g["cm_list_thb"],
    })
    out["cm_per_cup_drop_pct"] = (1 - out["cm_per_cup_promo"] / out["cm_per_cup_full"]) * 100
    return out


by_code = promo_cost(promo, ["base_code", "platform"])
display(by_code.round(2))
rc101_sku = promo_cost(promo[promo["base_code"] == "RC101"], "sku").reindex(ITEMS)
print("RC101 per item (both platforms):")
display(rc101_sku.round(2))
print(f"Total discount: {promo['discount_thb'].sum():,.0f} THB a year ({promo['discount_thb'].sum() / N_MONTHS:,.0f} a month)")

cups  pct_of_all_cups  discount_thb_year  \
base_code platform                                              
RC101     Grab       4997             0.81           41473.35   
          LINE MAN   9584             1.55           74620.20   
RC102     Grab      14753             2.39          150422.60   
          LINE MAN  19048             3.08          197926.40   
RC103     Grab       8610             1.39           52336.08   
          LINE MAN  11261             1.82           68767.80   

                    discount_thb_per_month  discount_pct  cm_per_cup_full  \
base_code platform                                                          
RC101     Grab                     3456.11          15.0            15.75   
          LINE MAN                 6218.35          15.0            12.73   
RC102     Grab                    12535.22          20.0            16.02   
          LINE MAN                16493.87          20.0            13.12   
RC103     Grab                     4361.34          12.0            15.41   
          LINE MAN                 5730.65          12.0            12.15   

                    cm_per_cup_promo  cm_thb_promo  cm_thb_if_full_price  \
base_code platform                                                         
RC101     Grab                  9.45      47201.75              78721.50   
          LINE MAN              7.28      69788.51             122022.65   
RC102     Grab                  8.27     122050.19             236371.37   
          LINE MAN              5.84     111325.72             249874.20   
RC103     Grab                 10.79      92931.17             132706.59   
          LINE MAN              7.88      88729.02             136866.48   

                    cm_per_cup_drop_pct  
base_code platform                       
RC101     Grab                    40.04  
          LINE MAN                42.81  
RC102     Grab                    48.37  
          LINE MAN                55.45  
RC103     Grab                    29.97  
          LINE MAN                35.17

RC101 per item (both platforms):


,cups,pct_of_all_cups,discount_thb_year,discount_thb_per_month,discount_pct,cm_per_cup_full,cm_per_cup_promo,cm_thb_promo,cm_thb_if_full_price,cm_per_cup_drop_pct
sku,,,,,,,,,,
Watermelon,7088,1.15,47844.00,3987.00,15.0,13.73,8.88,62958.80,97330.48,35.31
Pineapple,1711,0.28,12575.85,1047.99,15.0,15.20,9.89,16924.80,26007.05,34.92
Guava,2488,0.40,18286.80,1523.90,15.0,14.36,9.07,22573.30,35718.48,36.80
PassionFruit,1830,0.30,17842.50,1486.88,15.0,12.84,5.80,10611.15,23490.51,54.83
MixedBerryPremium,1464,0.24,19544.40,1628.70,15.0,12.43,2.68,3922.21,18197.63,78.45


Total discount: 585,546 THB a year (48,796 a month)


In [5]:
plot = by_code.reset_index()
plot["label"] = plot["base_code"] + " " + plot["platform"]
long = plot.melt(id_vars="label", value_vars=["cm_per_cup_full", "cm_per_cup_promo"],
                 var_name="price", value_name="cm_per_cup")
long["price"] = long["price"].map({"cm_per_cup_full": "At list price", "cm_per_cup_promo": "At promo price"})
fig = px.bar(long, x="label", y="cm_per_cup", color="price", barmode="group",
             color_discrete_map={"At list price": "#86b6ef", "At promo price": "#2a78d6"},
             text=long["cm_per_cup"].map("{:.1f}".format),
             labels={"cm_per_cup": "CM per cup (THB)", "label": "", "price": ""})
fig.update_traces(textposition="outside", cliponaxis=False, marker_line_color="white", marker_line_width=2,
                  hovertemplate="%{x}: %{y:.2f} THB a cup<extra>%{fullData.name}</extra>")
d102 = by_code.loc["RC102", "cm_per_cup_drop_pct"]
d101 = by_code.loc["RC101", "cm_per_cup_drop_pct"]
fig.update_layout(height=430, title=subtitle(
    f"A 20% RC102 cut removes {d102.min():.0f}-{d102.max():.0f}% of CM per cup; "
    f"RC101's 15% cut removes {d101.min():.0f}-{d101.max():.0f}%",
    "CM per cup on promo cups: at list price vs at the promo price (same cups, same week costs)"))
fig.show()

**Promos cost 585k THB of discount a year (49k a month); RC102 is the biggest at 348k.** RC102 takes CM per cup down 48-55%, RC101 40-43%, RC103 30-35%: the % lost in CM is 2.4-2.9x the % discount.

LINE MAN loses more than Grab on every code, because its 30% commission leaves less CM to start with. Berry on RC101 keeps 5.72 THB a cup on Grab and -0.45 on LINE MAN.

## 3. What lift does each promo need to break even?

In [6]:
detail = promo_cost(promo, ["base_code", "platform", "sku"])
elasticity = pd.read_csv(OUT / "elasticity.csv")
main_el = elasticity[elasticity["estimate"].str.startswith("1")].set_index("sku")["elasticity"]
disc = codes.set_index("base_code")["discount_pct"]


def add_lift(t):
    """Break-even lift and the lift implied by the lesson-6 main elasticity."""
    t = t.copy()
    loses = t["cm_per_cup_promo"] <= 0
    t["breakeven_lift_pct"] = np.where(loses, np.nan, (t["cm_per_cup_full"] / t["cm_per_cup_promo"] - 1) * 100)
    t["status"] = np.where(loses, "loses money on every cup", "")
    return t


detail = add_lift(detail)
idx = detail.index.to_frame(index=False)
detail["elasticity"] = idx["sku"].map(main_el).to_numpy()
detail["implied_lift_pct"] = ((1 - idx["base_code"].map(disc).to_numpy()) ** detail["elasticity"] - 1) * 100
detail["lift_gap_pct"] = detail["breakeven_lift_pct"] - detail["implied_lift_pct"]
detail = detail.reindex(ITEMS, level="sku")
with pd.option_context("display.max_rows", None):
    display(detail[["cups", "cm_per_cup_full", "cm_per_cup_promo", "breakeven_lift_pct", "implied_lift_pct", "status"]].round(1))

by_code = add_lift(by_code)
# Code-level implied lift = cup-weighted average of the items' implied lifts.
w = detail.assign(w=detail["implied_lift_pct"] * detail["cups"]).groupby(["base_code", "platform"])[["w", "cups"]].sum()
by_code["implied_lift_pct"] = w["w"] / w["cups"]
display(by_code[["cups", "cm_per_cup_full", "cm_per_cup_promo", "breakeven_lift_pct", "implied_lift_pct"]].round(1))
covered = detail[detail["implied_lift_pct"] >= detail["breakeven_lift_pct"]]
print(f"Code x platform x item rows where the implied lift covers break-even: {len(covered)} of {len(detail)}")
print(f"Rows that lose money on every cup: {(detail['status'] != '').sum()}")

cups  cm_per_cup_full  cm_per_cup_promo  \
base_code platform sku                                                          
RC101     Grab     Watermelon         2175             15.7              10.6   
                   Pineapple           633             16.8              11.2   
                   Guava               781             16.5              10.9   
                   PassionFruit        666             13.8               6.4   
                   MixedBerryPremium   742             15.9               5.7   
          LINE MAN Watermelon         4913             12.8               8.1   
                   Pineapple          1078             14.3               9.1   
                   Guava              1707             13.4               8.2   
                   PassionFruit       1164             12.3               5.5   
                   MixedBerryPremium   722              8.9              -0.4   
RC102     Grab     Watermelon         6660             15.5               8.7   
                   Pineapple          3475             16.3               8.9   
                   Guava              2075             16.3               8.9   
                   PassionFruit       1911             17.4               7.5   
                   MixedBerryPremium   632             14.4               0.9   
          LINE MAN Watermelon         6688             12.9               6.6   
                   Pineapple          5329             13.6               6.7   
                   Guava              3358             13.6               6.8   
                   PassionFruit       2662             13.3               4.2   
                   MixedBerryPremium  1011             10.0              -2.5   
RC103     Grab     Watermelon         3279             15.2              11.1   
                   Pineapple          2560             17.0              12.5   
                   Guava              1577             16.1              11.6   
                   PassionFruit        850             13.6               7.7   
                   MixedBerryPremium   344              6.9              -1.2   
          LINE MAN Watermelon         4855             12.5               8.8   
                   Pineapple          2794             14.1              10.0   
                   Guava              1676             13.1               9.0   
                   PassionFruit       1531              9.3               3.9   
                   MixedBerryPremium   405              0.9              -6.5   

                                      breakeven_lift_pct  implied_lift_pct  \
base_code platform sku                                                       
RC101     Grab     Watermelon                       48.3              30.9   
                   Pineapple                        49.9              23.9   
                   Guava                            51.0              33.3   
                   PassionFruit                    116.5              22.3   
                   MixedBerryPremium               177.4              20.3   
          LINE MAN Watermelon                       58.2              30.9   
                   Pineapple                        56.3              23.9   
                   Guava                            62.6              33.3   
                   PassionFruit                    124.6              22.3   
                   MixedBerryPremium                 NaN              20.3   
RC102     Grab     Watermelon                       78.6              44.8   
                   Pineapple                        83.9              34.2   
                   Guava                            84.1              48.4   
                   PassionFruit                    131.7              31.9   
                   MixedBerryPremium              1538.8              28.9   
          LINE MAN Watermelon                       95.7              44.8   
                   Pineapple             

cups  cm_per_cup_full  cm_per_cup_promo  \
base_code platform                                             
RC101     Grab       4997             15.8               9.4   
          LINE MAN   9584             12.7               7.3   
RC102     Grab      14753             16.0               8.3   
          LINE MAN  19048             13.1               5.8   
RC103     Grab       8610             15.4              10.8   
          LINE MAN  11261             12.2               7.9   

                    breakeven_lift_pct  implied_lift_pct  
base_code platform                                        
RC101     Grab                    66.8              27.7  
          LINE MAN                74.8              28.7  
RC102     Grab                    93.7              40.4  
          LINE MAN               124.5              39.8  
RC103     Grab                    42.8              21.4  
          LINE MAN                54.3              21.4

Code x platform x item rows where the implied lift covers break-even: 0 of 30
Rows that lose money on every cup: 4


In [7]:
plot = by_code.reset_index()
plot["label"] = plot["base_code"] + " " + plot["platform"]
fig = go.Figure()
fig.add_trace(go.Bar(x=plot["breakeven_lift_pct"], y=plot["label"], orientation="h", name="Needed to break even",
                     marker=dict(color="#86b6ef", line=dict(color="white", width=2)),
                     text=plot["breakeven_lift_pct"].map("{:.0f}%".format), textposition="outside", cliponaxis=False,
                     hovertemplate="%{y}: needs +%{x:.0f}% cups<extra></extra>"))
fig.add_trace(go.Scatter(x=plot["implied_lift_pct"], y=plot["label"], mode="markers", name="Implied by elasticity",
                         marker=dict(size=13, color="#e34948", symbol="diamond"),
                         hovertemplate="%{y}: elasticity implies +%{x:.0f}%<extra></extra>"))
fig.update_yaxes(autorange="reversed")
fig.update_xaxes(title="Extra cups vs a normal day (%)", range=[0, plot["breakeven_lift_pct"].max() * 1.2])
fig.update_layout(height=420, legend=dict(orientation="h", y=-0.2), title=subtitle(
    f"No promo pays back on CM: they need +{plot['breakeven_lift_pct'].min():.0f}% to +{plot['breakeven_lift_pct'].max():.0f}% cups; "
    f"elasticity implies +{plot['implied_lift_pct'].min():.0f}% to +{plot['implied_lift_pct'].max():.0f}%",
    "Code x platform, all items. Bars = break-even lift; diamonds = lift from the lesson-6 main elasticity (first guess)."))
fig.show()

**No code x platform pays back: they need +43% (RC103 Grab) to +124% (RC102 LINE MAN) more cups; the lesson-6 elasticity implies only +21% to +40%.** 0 of 30 code x platform x item rows reach break-even.

Berry is the worst case: it loses money on every promo cup in 4 of 6 code x platform pairs, and RC102 on Grab needs +1,539%. Among old items, PassionFruit RC102 on LINE MAN needs the most (+218%).

Baseline and waste are measured in sections 5-7. ? New users (RC102's goal) cannot be measured: the data has no customer column.

## 4. Do the promo tables add up?

In [8]:
assert by_code["cups"].sum() == promo["units_sold"].sum()
assert np.isclose(by_code["discount_thb_year"].sum(), e["discount_thb"].sum()), "all discount sits on promo rows"
assert np.isclose(by_code["cm_thb_promo"].sum(), promo["cm_thb"].sum())
assert np.isclose(detail["cm_thb_promo"].sum(), by_code["cm_thb_promo"].sum())
assert np.isclose(rc101_sku["discount_thb_year"].sum(), by_code.loc["RC101", "discount_thb_year"].sum())
assert e.loc[e["base_code"] == STANDARD, "discount_thb"].abs().max() < 1e-9
assert e.loc[e["base_code"] == STANDARD, "units_sold"].sum() + promo["units_sold"].sum() == TOTAL_CUPS

promo_summary = pd.concat([
    by_code.reset_index().assign(level="code_platform", sku="All"),
    detail.reset_index().assign(level="code_platform_sku"),
    rc101_sku.rename_axis("sku").reset_index().assign(level="rc101_sku", base_code="RC101", platform="All"),
], ignore_index=True)
cols = ["level", "base_code", "platform", "sku"]
promo_summary = promo_summary[cols + [c for c in promo_summary.columns if c not in cols]]
promo_summary.to_csv(OUT / "promo_summary.csv", index=False, encoding="utf-8")
ps = pd.read_csv(OUT / "promo_summary.csv")
for level in ["code_platform", "code_platform_sku"]:
    assert np.isclose(ps.loc[ps["level"] == level, "discount_thb_year"].sum(), e["discount_thb"].sum())
print(f"promo_summary.csv: {len(ps)} rows {ps['level'].value_counts().to_dict()} | "
      f"discount {e['discount_thb'].sum():,.0f} THB = orders_enriched")

promo_summary.csv: 41 rows {'code_platform_sku': 30, 'code_platform': 6, 'rc101_sku': 5} | discount 585,546 THB = orders_enriched


Promo cups, discount and CM add back to `orders_enriched` (68,253 cups, 585k THB discount). `promo_summary.csv` has 3 levels; filter on `level` before summing.

## 5. What would promo days have sold without the promo?

Slot = date x hour x kitchen x sku x platform, cups summed, hours with no sale = 0. Baseline = mean cups of normal slots with the same kitchen, sku, platform, month, weekday and hour. Normal = not a promo day, not the day after one, and not another item on an RC101 day. Promo days in Songkran week (13-19 Apr) are left out of the measurement: the holiday lifts every item, so they have no fair baseline.

In [9]:
from src.load_clean import load_tables, week_monday

tables = load_tables()
KITCHENS = ["Pattaya_Central", "Pattaya_Beach", "BKK_Sukhumvit", "BKK_Ladprao"]
SLOT = ["date", "hour", "kitchen", "sku", "platform_code"]
CELL = ["kitchen", "sku", "platform_code", "month", "weekday", "hour"]
WCELL = ["kitchen", "sku", "month", "weekday"]
SONGKRAN = (pd.Timestamp("2026-04-13"), pd.Timestamp("2026-04-19"))
PLAT = {"LM": "LINE MAN", "GB": "Grab"}

# Full slot grid: every hour of every day the item is on sale, so empty hours count as 0 cups.
sold = e.groupby(SLOT, as_index=False).agg(cups=("units_sold", "sum"), cm=("cm_thb", "sum"))
grid = []
for s, (d0, d1) in e.groupby("sku")["date"].agg(["min", "max"]).iterrows():
    grid.append(pd.MultiIndex.from_product([pd.date_range(d0, d1), range(24), KITCHENS, [s], ["LM", "GB"]],
                                           names=SLOT).to_frame(index=False))
slots = pd.concat(grid, ignore_index=True).merge(sold, on=SLOT, how="left", validate="one_to_one")
slots[["cups", "cm"]] = slots[["cups", "cm"]].fillna(0)
assert slots["cups"].sum() == TOTAL_CUPS and np.isclose(slots["cm"].sum(), e["cm_thb"].sum())

# Promo calendar per item-day, and the day after each promo day (if it is not a promo day itself).
promo_day = promo[["date", "sku", "base_code"]].drop_duplicates().rename(columns={"base_code": "code"})
assert not promo_day.duplicated(["date", "sku"]).any()
after = promo_day.assign(date=promo_day["date"] + pd.Timedelta(days=1)).rename(columns={"code": "after_code"})
after = after.merge(promo_day[["date", "sku"]], on=["date", "sku"], how="left", indicator=True)
after = after[after["_merge"] == "left_only"].drop(columns="_merge")
rc101_dates = promo_day.loc[promo_day["code"] == "RC101", ["date", "sku"]].rename(columns={"sku": "rc101_sku"})


def add_calendar(df):
    """Promo code, day-after code, RC101 'other item' flag, month and weekday."""
    df = (df.merge(promo_day, on=["date", "sku"], how="left").merge(after, on=["date", "sku"], how="left")
          .merge(rc101_dates, on="date", how="left"))
    df["other_on_rc101"] = df["rc101_sku"].notna() & df["code"].isna()
    df["month"] = df["date"].dt.strftime("%Y-%m")
    df["weekday"] = df["date"].dt.dayofweek
    # Rows that belong to a promo day in Songkran week: the promo item, other items that day, or the day after.
    in_sk = df["date"].between(*SONGKRAN)
    day_after_only = df["after_code"].notna() & df["code"].isna() & ~df["other_on_rc101"]
    df["songkran_event"] = ((df["code"].notna() | df["other_on_rc101"]) & in_sk) | \
        (day_after_only & (df["date"] - pd.Timedelta(days=1)).between(*SONGKRAN))
    return df


slots = add_calendar(slots)
comm = tables["platform_commission_rate"].set_index("platform_code")["commission_rate"]
uc = tables["unit_cost"].set_index("sku")
fc = tables["weekly_fruit_cost"].set_index(["week_start", "sku"])["fruit_cost_per_cup_thb"]
wk = week_monday(slots["date"])
# CM per cup at list price for every slot, from the week's fruit cost (also for slots with 0 cups).
slots["cm_list_per_cup"] = (uc.loc[slots["sku"], "base_price_thb"].to_numpy() * (1 - comm.loc[slots["platform_code"]].to_numpy())
                            - fc.loc[list(zip(wk, slots["sku"]))].to_numpy()
                            - uc.loc[slots["sku"], "pack_labor_per_cup_thb"].to_numpy())
assert slots["cm_list_per_cup"].notna().all()
slots["platform"] = slots["platform_code"].map(PLAT)
normal = slots["code"].isna() & slots["after_code"].isna() & ~slots["other_on_rc101"]

waste = add_calendar(tables["waste_daily"].copy())
waste_normal = waste["code"].isna() & waste["after_code"].isna() & ~waste["other_on_rc101"]

left_out = promo_day[promo_day["date"].between(*SONGKRAN)]
print(f"Slots: {len(slots):,} | promo slots: {slots['code'].notna().sum():,} | "
      f"Songkran-week promo item-days left out: {len(left_out)} {left_out.to_dict('records')}")

Slots: 315,648 | promo slots: 27,840 | Songkran-week promo item-days left out: 1 [{'date': Timestamp('2026-04-16 00:00:00'), 'sku': 'Guava', 'code': 'RC101'}]


In [10]:
def ev_id(df, code_col, sku_col, date_col="date"):
    """One promo event: RC101 = one item-day, RC102/RC103 = one date for all items."""
    d = df[date_col].dt.strftime("%Y-%m-%d")
    return np.where(df[code_col] == "RC101", df[code_col] + " " + d + " " + df[sku_col], df[code_col] + " " + d)


def measure(drop_songkran_baseline=False):
    """Promo, waste, spillover and pull-forward effects vs the baseline. Songkran-week events are left out."""
    pool = slots[normal]
    wpool = waste[waste_normal]
    if drop_songkran_baseline:
        pool = pool[~pool["date"].between(*SONGKRAN)]
        wpool = wpool[~wpool["date"].between(*SONGKRAN)]
    s = slots.join(pool.groupby(CELL)["cups"].mean().rename("baseline_cups"), on=CELL)
    w = waste.join(wpool.groupby(WCELL)[["units_wasted", "est_waste_cost_thb"]].mean().add_prefix("baseline_"), on=WCELL)
    # Positive waste effect = less waste than a normal day (a saving).
    w["waste_effect_thb"] = w["baseline_est_waste_cost_thb"] - w["est_waste_cost_thb"]
    keep = ~s["songkran_event"]
    wkeep = ~w["songkran_event"]

    ps = s[s["code"].notna() & keep].copy()
    ps["inc"] = ps["cm"] - ps["baseline_cups"] * ps["cm_list_per_cup"]
    ps["event"] = ev_id(ps, "code", "sku")
    # Spillover: other items on RC101 days (cups and their waste). Pull-forward: the day after, same item,
    # unless that slot is already counted as spillover.
    sp = s[s["other_on_rc101"] & keep].copy()
    sp["effect"] = (sp["cups"] - sp["baseline_cups"]) * sp["cm_list_per_cup"]
    sp["event"] = "RC101 " + sp["date"].dt.strftime("%Y-%m-%d") + " " + sp["rc101_sku"]
    pf = s[s["after_code"].notna() & s["code"].isna() & ~s["other_on_rc101"] & keep].copy()
    pf["effect"] = (pf["cups"] - pf["baseline_cups"]) * pf["cm_list_per_cup"]
    pf["event"] = ev_id(pf.assign(day=pf["date"] - pd.Timedelta(days=1)), "after_code", "sku", "day")

    wp = w[w["code"].notna() & wkeep].copy()
    wp["event"] = ev_id(wp, "code", "sku")
    wsp = w[w["other_on_rc101"] & wkeep].copy()
    wsp["event"] = "RC101 " + wsp["date"].dt.strftime("%Y-%m-%d") + " " + wsp["rc101_sku"]

    # Side effects per event x code x item x platform. Waste has no platform: split by the event's promo cups.
    key = ["event", "code", "sku", "platform"]
    cups_ev = ps.groupby(key)["cups"].sum()
    share = cups_ev / cups_ev.groupby(["event", "sku"]).transform("sum")
    w_item = wp.groupby(["event", "sku"])["waste_effect_thb"].sum()
    waste_ev = (share * w_item.reindex(share.droplevel(["code", "platform"]).index).to_numpy()).rename("waste")
    ev_share = cups_ev.groupby(["event", "platform"]).sum()
    ev_share = ev_share[ev_share.index.get_level_values("event").str.startswith("RC101")]
    ev_share = ev_share / ev_share.groupby("event").transform("sum")
    spill_w = (ev_share * wsp.groupby("event")["waste_effect_thb"].sum().reindex(ev_share.index.get_level_values("event")).to_numpy())
    spill_cups = sp.groupby(["event", "rc101_sku", "platform"])["effect"].sum().droplevel("rc101_sku")
    spill_ev = spill_cups.add(spill_w, fill_value=0).rename("spill")
    spill_ev.index = pd.MultiIndex.from_tuples([(ev, "RC101", ev.split(" ")[2], pl) for ev, pl in spill_ev.index], names=key)
    pull_ev = pf.rename(columns={"after_code": "pcode"}).groupby(["event", "pcode", "sku", "platform"])["effect"].sum().rename("pull")
    pull_ev.index.names = key
    side = pd.concat([waste_ev, spill_ev, pull_ev], axis=1).fillna(0)
    inc_ev = ps.groupby(key).agg(cups=("cups", "sum"), baseline_cups=("baseline_cups", "sum"), cm=("cm", "sum"), inc=("inc", "sum"))
    out = inc_ev.join(side, how="outer").fillna(0)
    out["net"] = out[["inc", "waste", "spill", "pull"]].sum(axis=1)
    return {"slots": s, "promo_slots": ps, "waste": w, "ev": out}


M = measure()
ps_main = M["promo_slots"]
unmatched = int(ps_main["baseline_cups"].isna().sum())
days_per_cell = slots[normal].groupby(["sku", "month", "weekday"])["date"].nunique()
cell_days = ps_main[["sku", "month", "weekday"]].drop_duplicates().join(days_per_cell.rename("normal_days"), on=["sku", "month", "weekday"])
print(f"Unmatched promo slots: {unmatched} | promo item-month-weekday cells with 1 normal day: "
      f"{(cell_days['normal_days'] == 1).sum()}, with 2: {(cell_days['normal_days'] == 2).sum()}, of {len(cell_days)}")
assert unmatched == 0

Unmatched promo slots: 0 | promo item-month-weekday cells with 1 normal day: 10, with 2: 49, of 124


315,648 slots; 27,840 on promo days. One promo item-day falls in Songkran week (RC101 Guava, 16 Apr) and is left out. Every measured promo slot finds a baseline; 10 of 124 promo item-month-weekday cells rest on 1 normal day and 49 on 2.

## 6. What did each promo really add?

In [11]:
def by_row(ev, by):
    g = ev.groupby(level=by)[["cups", "baseline_cups", "cm", "inc", "waste", "spill", "pull", "net"]].sum()
    g["actual_lift_pct"] = (g["cups"] / g["baseline_cups"] - 1) * 100
    return g


inc = by_row(M["ev"], ["code", "platform"])
inc["breakeven_lift_pct"] = by_code["breakeven_lift_pct"].to_numpy()
inc["implied_lift_pct"] = by_code["implied_lift_pct"].to_numpy()
display(inc[["cups", "baseline_cups", "actual_lift_pct", "implied_lift_pct", "breakeven_lift_pct", "inc"]].round(1))

inc_rc101 = by_row(M["ev"].xs("RC101", level="code", drop_level=False), ["sku"]).reindex(ITEMS)
inc_rc101["breakeven_lift_pct"] = (rc101_sku["cm_per_cup_full"] / rc101_sku["cm_per_cup_promo"] - 1) * 100
print("RC101 per item, both platforms (Songkran-week day left out):")
display(inc_rc101[["cups", "baseline_cups", "actual_lift_pct", "breakeven_lift_pct", "inc"]].round(1))

measured_cups = inc["cups"].sum()
assert measured_cups + promo.loc[promo["date"].between(*SONGKRAN), "units_sold"].sum() == promo["units_sold"].sum()

cups  baseline_cups  actual_lift_pct  implied_lift_pct  \
code  platform                                                              
RC101 Grab       4997.0         5027.0             -0.6              27.7   
      LINE MAN   8823.0         5376.7             64.1              28.7   
RC102 Grab      14753.0         8938.2             65.1              40.4   
      LINE MAN  19048.0        10430.5             82.6              39.8   
RC103 Grab       8610.0         8004.2              7.6              21.4   
      LINE MAN  11261.0         9308.8             21.0              21.4   

                breakeven_lift_pct      inc  
code  platform                               
RC101 Grab                    66.8 -33866.5  
      LINE MAN                74.8  -2077.9  
RC102 Grab                    93.7 -22159.0  
      LINE MAN               124.5 -25381.3  
RC103 Grab                    42.8 -29135.0  
      LINE MAN                54.3 -25114.9

RC101 per item, both platforms (Songkran-week day left out):


,cups,baseline_cups,actual_lift_pct,breakeven_lift_pct,inc
sku,,,,,
Watermelon,7088.0,5267.5,34.6,54.6,-11762.4
Pineapple,1711.0,1290.0,32.6,53.7,-3713.2
Guava,1727.0,1301.1,32.7,58.2,-3280.2
PassionFruit,1830.0,1436.9,27.4,121.4,-8137.5
MixedBerryPremium,1464.0,1108.2,32.1,364.0,-9051.0


**RC102 lifts +75% (both platforms), about twice its price-implied +40%: double dates bring demand on top of the 20% cut.** RC103 lifts +15%, below its implied +21%. Both stay below their break-even lifts.

RC101 per item lifts +27% to +35% (Guava +33% once the Songkran day is out) against item break-evens of +54% to +364%. Lifts by platform are not reliable for any code: see the check below.

In [12]:
# Check 1: item-days where the whole day sold on one platform only.
n_plat = e.groupby(["date", "sku"])["platform_code"].nunique()
one_platform = set(n_plat[n_plat == 1].index)
pdays = promo_day.assign(one_platform=[(d, s) in one_platform for d, s in zip(promo_day["date"], promo_day["sku"])])
normal_item_days = e[e["base_code"] == STANDARD].groupby(["date", "sku"])["platform_code"].nunique()
platform_check = pdays.groupby("code").agg(item_days=("sku", "size"), one_platform_days=("one_platform", "sum"))
platform_check.loc[f"{STANDARD} (normal days)"] = [len(normal_item_days), int((normal_item_days == 1).sum())]
platform_check["one_platform_pct"] = platform_check["one_platform_days"] / platform_check["item_days"] * 100
assert platform_check.loc[PROMOS, "item_days"].sum() == len(pdays)
# The app reads this file for the data limits on page 6.
platform_check.rename_axis("code").reset_index().to_csv(OUT / "promo_platform_check.csv", index=False, encoding="utf-8")
display(platform_check.round(1))

ev_chk = M["ev"].copy()
ev_idx = ev_chk.index.to_frame(index=False)
ev_dates = pd.to_datetime(ev_idx["event"].str.split(" ").str[1])
ev_chk["one_platform"] = [(d, s) in one_platform for d, s in zip(ev_dates, ev_idx["sku"])]
compare = pd.concat({
    "all item-days": by_row(ev_chk, ["code", "platform"]),
    "one-platform item-days out": by_row(ev_chk[~ev_chk["one_platform"]], ["code", "platform"]),
}, axis=1)
compare = compare.loc[:, (slice(None), ["actual_lift_pct", "net"])]
compare.loc[:, (slice(None), "net")] = compare.loc[:, (slice(None), "net")] / N_MONTHS
display(compare.round(1))

# Check 2: leave out promo item-days whose baseline cell (item, month, weekday) has only 1 normal day.
thin = set(map(tuple, cell_days.loc[cell_days["normal_days"] == 1, ["sku", "month", "weekday"]].to_numpy()))
ev_months = ev_dates.dt.strftime("%Y-%m")
ev_chk["thin"] = [(s, m, d.dayofweek) in thin for s, m, d in zip(ev_idx["sku"], ev_months, ev_dates)]
by_code_net = pd.DataFrame({
    "all": ev_chk.groupby(level="code")["net"].sum() / N_MONTHS,
    "one-platform out": ev_chk[~ev_chk["one_platform"]].groupby(level="code")["net"].sum() / N_MONTHS,
    "thin cells out": ev_chk[~ev_chk["thin"]].groupby(level="code")["net"].sum() / N_MONTHS,
})
print(f"Promo event rows on thin baseline cells: {ev_chk['thin'].sum()} of {len(ev_chk)}")
display(by_code_net.round(0))

,item_days,one_platform_days,one_platform_pct
code,,,
RC101,32,11,34.4
RC102,54,7,13.0
RC103,59,4,6.8
RC000 (normal days),1499,198,13.2


all item-days one-platform item-days out all item-days  \
               actual_lift_pct            actual_lift_pct           net   
code  platform                                                            
RC101 Grab                -0.6                       31.4       -4040.2   
      LINE MAN            64.1                       39.1         424.8   
RC102 Grab                65.1                       79.4       -2345.3   
      LINE MAN            82.6                       70.9       -3970.6   
RC103 Grab                 7.6                       17.3       -3176.5   
      LINE MAN            21.0                       13.1       -1531.7   

               one-platform item-days out  
                                      net  
code  platform                             
RC101 Grab                        -1124.5  
      LINE MAN                    -1376.0  
RC102 Grab                        -1266.8  
      LINE MAN                    -4051.9  
RC103 Grab                        -2248.5  
      LINE MAN                    -1950.8

Promo event rows on thin baseline cells: 28 of 288


,all,one-platform out,thin cells out
code,,,
RC101,-3615.0,-2500.0,-2750.0
RC102,-6316.0,-5319.0,-4205.0
RC103,-4708.0,-4199.0,-4708.0


**On 11 of 32 RC101 item-days the item sold on one platform only (34%, vs 13% on normal days); RC102 7 of 54, RC103 4 of 59.** Without those item-days RC101 lifts +31% (Grab) and +39% (LINE MAN), and RC102 and RC103 swap which platform lifts more. Read lifts per code; the net stays negative for every code and platform.

Leaving out promo days on thin baseline cells (1 normal day) does not change any sign. ? Ask the data owner how the export assigns orders to platforms.

## 7. Do promos change waste?

In [13]:
wv = M["waste"]
wv_promo = wv[wv["code"].notna() & ~wv["songkran_event"]]
waste_code = wv_promo.groupby("code")[["units_wasted", "baseline_units_wasted", "est_waste_cost_thb",
                                         "baseline_est_waste_cost_thb", "waste_effect_thb"]].sum()
display(waste_code.round(0))
display(wv_promo.groupby(["code", "sku"])[["units_wasted", "baseline_units_wasted", "waste_effect_thb"]].sum()
        .reindex(ITEMS, level="sku").round(0))
wv_spill = wv[wv["other_on_rc101"] & ~wv["songkran_event"]]
print(f"Waste of other items on RC101 days: {wv_spill['units_wasted'].sum():,.0f} vs baseline "
      f"{wv_spill['baseline_units_wasted'].sum():,.0f} cups, effect {wv_spill['waste_effect_thb'].sum():,.0f} THB (in spillover)")
assert np.isclose(inc["waste"].sum(), wv_promo["waste_effect_thb"].sum())

,units_wasted,baseline_units_wasted,est_waste_cost_thb,baseline_est_waste_cost_thb,waste_effect_thb
code,,,,,
RC101,1007.0,760.0,29944.0,22323.0,-7621.0
RC102,2045.0,1131.0,48625.0,26566.0,-22059.0
RC103,1087.0,987.0,26473.0,24948.0,-1525.0


units_wasted  baseline_units_wasted  waste_effect_thb
code  sku                                                                     
RC101 Watermelon                245.0                  178.0            -885.0
      Pineapple                  76.0                   62.0            -278.0
      Guava                      99.0                   78.0            -324.0
      PassionFruit              164.0                  148.0            -484.0
      MixedBerryPremium         423.0                  294.0           -5650.0
RC102 Watermelon                448.0                  247.0           -2731.0
      Pineapple                 394.0                  233.0           -2706.0
      Guava                     323.0                  184.0           -2103.0
      PassionFruit              451.0                  244.0           -5342.0
      MixedBerryPremium         429.0                  223.0           -9176.0
RC103 Watermelon                283.0                  227.0            -780.0
      Pineapple                 235.0                  206.0            -395.0
      Guava                     166.0                  158.0             -82.0
      PassionFruit              228.0                  211.0            -489.0
      MixedBerryPremium         175.0                  185.0             220.0

Waste of other items on RC101 days: 2,282 vs baseline 2,293 cups, effect 653 THB (in spillover)


**Promo days waste more, not less: RC102 days throw away 2,045 cups vs 1,131 on comparable days (+81%), 22k THB more a year; RC101 days 7.6k more.** Prep seems to follow expected demand and overshoot it. Berry takes the biggest hit (RC102 -9.2k, RC101 -5.7k); RC103 barely moves waste (-1.5k).

Other items on RC101 days waste the same as on normal days (2,282 vs 2,293 cups). Waste is split to platforms by promo cups (an allocation).

## 8. Do promos move sales to other items or other days?

In [14]:
display(inc[["inc", "waste", "spill", "pull", "net"]].round(0))
inc["net_per_month"] = inc["net"] / N_MONTHS
check = by_row(measure(drop_songkran_baseline=True)["ev"], ["code", "platform"])
inc["net_per_month_songkran_out_of_baseline"] = check["net"] / N_MONTHS
display(inc[["net_per_month", "net_per_month_songkran_out_of_baseline"]].round(0))
print(f"All promos (measured events): net {inc['net'].sum():,.0f} THB a year ({inc['net'].sum() / N_MONTHS:,.0f} a month); "
      f"Songkran week out of every baseline: {check['net'].sum() / N_MONTHS:,.0f} a month")
skipped = slots[slots["songkran_event"] & slots["code"].notna()].join(
    slots[normal].groupby(CELL)["cups"].mean().rename("b"), on=CELL)
print(f"Left out, RC101 Guava 16 Apr: {skipped['cups'].sum():,.0f} cups vs baseline {skipped['b'].sum():,.0f}; "
      "every other item also sold far above baseline that day")

inc    waste   spill     pull      net
code  platform                                            
RC101 Grab     -33867.0  -3337.0 -4722.0  -6557.0 -48483.0
      LINE MAN  -2078.0  -4284.0  4971.0   6489.0   5097.0
RC102 Grab     -22159.0 -10004.0     0.0   4020.0 -28143.0
      LINE MAN -25381.0 -12055.0     0.0 -10211.0 -47647.0
RC103 Grab     -29135.0   -720.0     0.0  -8263.0 -38118.0
      LINE MAN -25115.0   -805.0     0.0   7540.0 -18380.0

net_per_month  net_per_month_songkran_out_of_baseline
code  platform                                                       
RC101 Grab            -4040.0                                 -2315.0
      LINE MAN          425.0                                  -892.0
RC102 Grab            -2345.0                                 -1372.0
      LINE MAN        -3971.0                                 -4557.0
RC103 Grab            -3177.0                                 -3177.0
      LINE MAN        -1532.0                                 -1532.0

All promos (measured events): net -175,674 THB a year (-14,639 a month); Songkran week out of every baseline: -13,843 a month
Left out, RC101 Guava 16 Apr: 761 cups vs baseline 348; every other item also sold far above baseline that day


**After waste, spillover and pull-forward, every code loses CM: RC101 -3.6k, RC102 -6.3k, RC103 -4.7k THB a month (176k a year).** By platform 5 of 6 pairs are negative; the exception, RC101 LINE MAN, comes from the one-platform item-days and turns negative without them.

With the Songkran day left out, spillover on RC101 days is close to 0 (+0.2k THB a year) and pull-forward flips sign by platform, so neither pays for a promo. With Songkran week out of every baseline the signs do not change.

## 9. How sure are we?

In [15]:
events = M["ev"].groupby(level=["event", "code"])["net"].sum().reset_index(level="code")
assert np.isclose(events["net"].sum(), inc["net"].sum())
rng = np.random.default_rng(7)
B = 2000
summary = []
for c, g in events.groupby("code"):
    vals = g["net"].to_numpy()
    boot = np.array([rng.choice(vals, len(vals), replace=True).sum() for _ in range(B)]) / N_MONTHS
    summary.append({"code": c, "events": len(vals), "events_positive": int((vals > 0).sum()),
                    "net_per_month": vals.sum() / N_MONTHS, "p5_per_month": np.percentile(boot, 5),
                    "p95_per_month": np.percentile(boot, 95)})
uncertainty = pd.DataFrame(summary).set_index("code")
display(uncertainty.round(0))
display(events.sort_values("net", ascending=False).head(5).round(0))

,events,events_positive,net_per_month,p5_per_month,p95_per_month
code,,,,,
RC101,31,7,-3615.0,-5122.0,-2196.0
RC102,12,0,-6316.0,-8380.0,-4697.0
RC103,13,0,-4708.0,-5277.0,-4162.0


,code,net
event,,
RC101 2026-01-18 Guava,RC101,1477.0
RC101 2025-10-15 Guava,RC101,1471.0
RC101 2026-06-01 MixedBerryPremium,RC101,1082.0
RC101 2026-05-04 Watermelon,RC101,985.0
RC101 2025-09-05 Pineapple,RC101,694.0


**All three codes have a 90% range below 0: RC101 -5.1k to -2.2k, RC102 -8.4k to -4.7k, RC103 -5.3k to -4.2k THB a month.** RC102 is negative on all 12 days, RC103 on all 13, RC101 on 24 of 31 measured item-days.

The bootstrap shows how much the result depends on which days got a promo. It does not test the baseline or the waste split.

## 10. Which promos earn back their cost?

In [16]:
by_code_row = by_row(M["ev"], ["code"])
rc101_row = by_row(M["ev"].xs("RC101", level="code", drop_level=False), ["sku"])
plot = pd.concat([
    rc101_row.reset_index().assign(label=lambda t: "RC101 " + t["sku"], code="RC101"),
    by_code_row.drop(index="RC101").reset_index().assign(label=lambda t: t["code"] + " (all items)"),
], ignore_index=True)
plot["net_per_month"] = plot["net"] / N_MONTHS
fig = px.scatter(plot, x="actual_lift_pct", y="net_per_month", size="cups", text="label", color="code",
                 color_discrete_map={"RC101": "#2a78d6", "RC102": "#eb6834", "RC103": "#1baf7a"}, size_max=45,
                 labels={"actual_lift_pct": "Actual lift vs baseline (%)", "net_per_month": "Net incremental CM per month (THB)",
                         "code": ""})
fig.update_traces(textposition="top center", marker=dict(line=dict(color="white", width=2)),
                  hovertemplate="%{text}<br>lift %{x:.0f}%<br>%{y:,.0f} THB a month<extra></extra>")
fig.add_hline(y=0, line_color="#52514e", line_width=1)
worst = plot.loc[plot["net_per_month"].idxmin()]
n_neg = int((plot["net_per_month"] < 0).sum())
fig.update_layout(height=470, showlegend=False, title=subtitle(
    f"{n_neg} of {len(plot)} promo rows lose CM after waste and side effects; "
    f"{worst['label']} loses the most ({abs(worst['net_per_month']):,.0f} THB a month) with a {worst['actual_lift_pct']:+.0f}% lift",
    "RC101 per item, RC102 and RC103 per code, both platforms. Size = promo cups. Songkran-week promo day left out."))
fig.show()

PLATFORM_NOTE = {"RC101": "platform split not reliable (one-platform item-days); use rc101_sku rows",
                 "RC102": "platform lifts swap order without one-platform item-days; use the code row for lift",
                 "RC103": "platform lifts swap order without one-platform item-days; use the code row for lift"}
promo_incremental = pd.concat([
    by_code_row.reset_index().assign(level="code", platform="All", sku="All", note=""),
    rc101_row.reset_index().assign(level="rc101_sku", code="RC101", platform="All", note=""),
    inc.reset_index().assign(level="code_platform", sku="All", note=lambda t: t["code"].map(PLATFORM_NOTE)),
], ignore_index=True)
cols = ["level", "code", "platform", "sku", "note"]
promo_incremental = promo_incremental[cols + [c for c in promo_incremental.columns if c not in cols]]
promo_incremental.to_csv(OUT / "promo_incremental.csv", index=False, encoding="utf-8")
pi = pd.read_csv(OUT / "promo_incremental.csv")
for level in ["code", "code_platform"]:
    assert np.isclose(pi.loc[pi["level"] == level, "net"].sum(), events["net"].sum())
assert np.isclose(pi.loc[pi["level"] == "rc101_sku", "net"].sum(), by_code_row.loc["RC101", "net"])
print(f"promo_incremental.csv: {len(pi)} rows {pi['level'].value_counts().to_dict()}")

promo_incremental.csv: 14 rows {'code_platform': 6, 'rc101_sku': 5, 'code': 3}


promo_incremental.csv has 3 levels (code, rc101_sku, code_platform); filter on `level` before summing. code_platform rows carry a note: their platform split is not reliable.

## 11. What should we do with each promo?

| Rule | Condition (in order) | Decision |
|---|---|---|
| K1 | CM per cup at promo price <= 0 | Kill |
| K2 | Net incremental CM < 0 and 90% range below 0 | Kill, unless a Modify option turns it positive |
| K3 | Net < 0 and range crosses 0, or fewer than 6 promo days | Modify + test if the best option is above 0, else Kill |
| K4 | Net > 0 and most days positive | Keep |
| K5 | RC102: new users cannot be measured | judge on CM only, confidence low |

Rows: RC101 per item (both platforms), RC102 and RC103 per platform; this assumes the platform-level setting is possible for these company-wide codes; the check below shows each code on both platforms together. Watermelon and PassionFruit are valued at their lesson-6 prices (49, 69). Berry stays at 89 until the delist review; any berry saving here is already inside a delist.

Modify assumptions: (a) a shallower discount keeps the measured non-price lift (double date, rainy day) and scales cups with the lesson-6 elasticity; (c) quiet hours only: promo cups in 6-10 and 22-5, other hours at baseline, no orders move into promo hours; (d) berry removed from the code; side effects scale with the extra cups. Each row takes the best of 3-5 options picked on the same data, so Modify values are optimistic until tested.

In [17]:
NEW_LIST = {"Watermelon": 49.0, "PassionFruit": 69.0}   # lesson 6 price decisions
NEW_DISC = {"RC101": 0.10, "RC102": 0.12, "RC103": 0.08}
QUIET = set(range(6, 11)) | {22, 23, 0, 1, 2, 3, 4, 5}  # Morning 6-10 and Late night 22-5 (daypart_share.csv)
KEY = ["event", "code", "sku", "platform"]

pslot = ps_main.copy()
pslot["c"] = pslot["platform_code"].map(comm)
pslot["list"] = pslot["sku"].map(uc["base_price_thb"])
pslot["d"] = pslot["code"].map(disc)
pslot["el"] = pslot["sku"].map(main_el)
pslot["cm_promo_pc"] = pslot["cm_list_per_cup"] - pslot["d"] * pslot["list"] * (1 - pslot["c"])
assert np.isclose((pslot["cups"] * pslot["cm_promo_pc"]).sum(), pslot["cm"].sum())
side_ev = M["ev"][["waste", "spill", "pull"]].sum(axis=1)


def scenario(new_disc=None, hours=None, reprice=True):
    """Net incremental CM per event x code x item x platform under a Modify option."""
    d = pslot.copy()
    if reprice:
        for s, p1 in NEW_LIST.items():
            m = d["sku"] == s
            f = (p1 / d.loc[m, "list"]) ** d.loc[m, "el"]
            gap = (p1 - d.loc[m, "list"]) * (1 - d.loc[m, "c"])
            d.loc[m, "cm_list_per_cup"] += gap
            d.loc[m, "cm_promo_pc"] += gap * (1 - d.loc[m, "d"])
            d.loc[m, ["cups", "baseline_cups"]] = d.loc[m, ["cups", "baseline_cups"]].mul(f, axis=0)
            d.loc[m, "list"] = p1
    a0 = d.groupby(KEY)[["cups", "baseline_cups"]].sum()
    if new_disc is not None:
        nd = d["code"].map(new_disc)
        d["cups"] = d["cups"] * ((1 - nd) / (1 - d["d"])) ** d["el"]
        d["cm_promo_pc"] = d["cm_list_per_cup"] - nd * d["list"] * (1 - d["c"])
    if hours is not None:
        off = ~d["hour"].isin(hours)  # no promo in these hours: baseline cups at list CM
        d.loc[off, "cups"] = d.loc[off, "baseline_cups"]
        d.loc[off, "cm_promo_pc"] = d.loc[off, "cm_list_per_cup"]
    d["inc"] = d["cups"] * d["cm_promo_pc"] - d["baseline_cups"] * d["cm_list_per_cup"]
    a1 = d.groupby(KEY).agg(inc=("inc", "sum"), cups=("cups", "sum"), base=("baseline_cups", "sum"))
    a1 = a1.reindex(a1.index.union(side_ev.index)).fillna(0)
    a0 = a0.reindex(a1.index).fillna(0)
    extra0 = a0["cups"] - a0["baseline_cups"]
    ratio = ((a1["cups"] - a1["base"]) / extra0).where(extra0.abs() > 1e-9, 1).clip(lower=0)
    side = side_ev.reindex(a1.index).fillna(0)
    if reprice:
        for s, p1 in NEW_LIST.items():
            m = a1.index.get_level_values("sku") == s
            side[m] *= (p1 / uc.loc[s, "base_price_thb"]) ** main_el[s]
    a1["net"] = a1["inc"] + side * ratio
    return a1


SCEN = {"now": scenario(reprice=False), "new_prices": scenario(), "a_shallower": scenario(new_disc=NEW_DISC),
        "c_quiet_hours": scenario(hours=QUIET), "ac_shallower_quiet": scenario(new_disc=NEW_DISC, hours=QUIET)}
assert np.isclose(SCEN["now"]["net"].sum(), inc["net"].sum())

In [18]:
def row_result(ev, code, plat, sku, drop_berry=False, B=2000, seed=11):
    """Net per month, 90% bootstrap range over promo days, and days positive for one decision row."""
    i = ev.index
    m = (i.get_level_values("code") == code)
    if plat != "All":
        m &= i.get_level_values("platform") == plat
    if sku != "All":
        m &= i.get_level_values("sku") == sku
    if drop_berry:
        m &= i.get_level_values("sku") != "MixedBerryPremium"
    vals = ev[m].groupby(level="event")["net"].sum().to_numpy()
    rng = np.random.default_rng(seed)
    boot = np.array([rng.choice(vals, len(vals)).sum() for _ in range(B)]) / N_MONTHS
    return {"net": vals.sum() / N_MONTHS, "p5": np.percentile(boot, 5), "p95": np.percentile(boot, 95),
            "days": len(vals), "days_positive": int((vals > 0).sum())}


OPTIONS = {  # name: (scenario, drop berry, change text)
    "a": ("a_shallower", False, "shallower discount ({d})"),
    "c": ("c_quiet_hours", False, "quiet hours only (6-10, 22-5)"),
    "ac": ("ac_shallower_quiet", False, "shallower discount ({d}) + quiet hours"),
    "d": ("new_prices", True, "remove berry"),
    "ad": ("a_shallower", True, "shallower discount ({d}) + remove berry"),
}
CONFIDENCE = {("RC101", "Watermelon"): "medium", ("RC101", "Pineapple"): "low", ("RC101", "Guava"): "low",
              ("RC101", "PassionFruit"): "medium", ("RC101", "MixedBerryPremium"): "high",
              ("RC102", "Grab"): "low", ("RC102", "LINE MAN"): "low",  # K5
              ("RC103", "Grab"): "high", ("RC103", "LINE MAN"): "high"}
RISK = {
    ("RC101", "Watermelon"): "best option is still below 0",
    ("RC101", "Pineapple"): "only 3 promo days; an hours-only promo was never run",
    ("RC101", "Guava"): "5 measured days; its best day was in Songkran week and is left out",
    ("RC101", "PassionFruit"): "best option is still below 0",
    ("RC101", "MixedBerryPremium"): "saving is inside a berry delist (lesson 6); do not add both",
    ("RC102", "Grab"): "new users not measured; April baseline; berry part overlaps a berry delist",
    ("RC102", "LINE MAN"): "new users not measured; stopping on LINE MAN (0) is the safe alternative; berry overlaps a delist",
    ("RC103", "Grab"): "no option above 0; berry part overlaps a berry delist",
    ("RC103", "LINE MAN"): "no option above 0; berry part overlaps a berry delist",
}
rows_def = [("RC101", "All", s) for s in ITEMS] + [(c, p, "All") for c in ["RC102", "RC103"] for p in ["Grab", "LINE MAN"]]

out = []
for code, plat, sku in rows_def:
    sel = pslot[(pslot["code"] == code) & ((pslot["platform"] == plat) | (plat == "All")) & ((pslot["sku"] == sku) | (sku == "All"))]
    cm_pc = sel["cm_promo_pc"] + sel["sku"].map(NEW_LIST).sub(sel["list"]).fillna(0) * (1 - sel["c"]) * (1 - sel["d"])
    r_now = row_result(SCEN["now"], code, plat, sku)
    r = row_result(SCEN["new_prices"], code, plat, sku)
    opts = {k: row_result(SCEN[s], code, plat, sku, drop_berry=db) for k, (s, db, _) in OPTIONS.items()}
    if sku != "All":
        opts.pop("d")
        opts.pop("ad")
    best_key = max(opts, key=lambda k: opts[k]["net"])
    best = opts[best_key]
    cm_per_cup = (sel["cups"] * cm_pc).sum() / sel["cups"].sum()
    change = OPTIONS[best_key][2].format(d=f"{NEW_DISC[code]:.0%}")
    if cm_per_cup <= 0:
        rule, decision = "K1", "Kill"
    elif r["net"] < 0 and r["p95"] < 0:
        rule, decision = ("K2 (Modify turns it positive)", "Modify + test") if best["net"] > 0 else ("K2", "Kill")
    elif (r["net"] < 0 and r["p95"] >= 0) or r["days"] < 6:
        rule, decision = ("K3", "Modify + test") if best["net"] > 0 else ("K3 (no option above 0)", "Kill")
    elif r["net"] > 0 and r["days_positive"] > r["days"] / 2:
        rule, decision = "K4", "Keep"
    else:
        rule, decision = ("none (judged as K3)", "Modify + test") if best["net"] > 0 else ("none (judged as K3)", "Kill")
    if decision == "Kill":
        change = "stop the berry bundle" if sku == "MixedBerryPremium" else "stop"
    key = (code, sku if code == "RC101" else plat)
    res = {"code": code, "platform": plat, "sku": sku, "rule": rule, "decision": decision, "change": change,
           "cm_per_cup_promo": cm_per_cup, "promo_days": r["days"], "days_positive": r["days_positive"],
           "net_per_month_now": r_now["net"], "net_per_month_new_prices": r["net"],
           "range_low_new_prices": r["p5"], "range_high_new_prices": r["p95"],
           "best_option": best_key, "best_option_net": best["net"],
           # (b) Grab only: on a LINE MAN row the promo stops (0); on a Grab row nothing changes.
           "option_b_grab_only": 0.0 if plat == "LINE MAN" else (r["net"] if plat == "Grab" else np.nan)}
    if decision == "Kill":
        res.update(kill_saving=-r["net"], modify_net=np.nan, modify_low=np.nan, modify_high=np.nan)
    else:
        res.update(kill_saving=0.0, modify_net=best["net"], modify_low=best["p5"], modify_high=best["p95"])
    res.update(confidence=CONFIDENCE[key], main_risk=RISK[key])
    out.append(res)
promo_decision = pd.DataFrame(out)

with pd.option_context("display.max_colwidth", None):
    display(promo_decision[["code", "platform", "sku", "rule", "decision", "change", "net_per_month_new_prices",
                            "range_low_new_prices", "range_high_new_prices", "kill_saving", "modify_net", "modify_low",
                            "modify_high", "option_b_grab_only", "confidence"]].round(0))

# Info only: RC103 on high waste-risk kitchen-days (baseline waste above the RC103 median).
r103 = M["waste"][(M["waste"]["code"] == "RC103")]
kd = r103.groupby(["date", "kitchen"])["baseline_est_waste_cost_thb"].sum()
high = kd[kd > kd.median()].index
sub = pslot[pd.MultiIndex.from_frame(pslot[["date", "kitchen"]]).isin(high) & (pslot["code"] == "RC103")]
inc_e = (sub["cups"] * sub["cm_promo_pc"] - sub["baseline_cups"] * sub["cm_list_per_cup"]).sum()
waste_e = r103.set_index(["date", "kitchen"]).loc[high, "waste_effect_thb"].sum()
print(f"(e) RC103 on {len(high)} of {len(kd)} high waste-risk kitchen-days: {(inc_e + waste_e) / N_MONTHS:,.0f} THB a month")

kill_total = promo_decision["kill_saving"].sum()
mod = promo_decision[promo_decision["decision"] == "Modify + test"]
mod_gain = (mod["modify_net"] - mod["net_per_month_new_prices"]).sum()
print(f"At lesson-6 prices promos make {promo_decision['net_per_month_new_prices'].sum():,.0f} THB a month. "
      f"Kill rows save {kill_total:,.0f}. Modify rows, if tests confirm: {mod_gain:+,.0f} "
      f"(best options sum {mod['modify_net'].sum():,.0f}, low {mod['modify_low'].sum():,.0f}, high {mod['modify_high'].sum():,.0f})")

# If the platform cannot be set separately, judge RC102 and RC103 on both platforms together.
for c in ["RC102", "RC103"]:
    rows = promo_decision[promo_decision["code"] == c]
    print(f"{c} both platforms: {rows['net_per_month_new_prices'].sum():,.0f} THB a month today, "
          f"best option {rows['best_option_net'].sum():,.0f}")

promo_decision.to_csv(OUT / "promo_decision.csv", index=False, encoding="utf-8")
assert len(pd.read_csv(OUT / "promo_decision.csv")) == 9

,code,platform,sku,rule,decision,change,net_per_month_new_prices,range_low_new_prices,range_high_new_prices,kill_saving,modify_net,modify_low,modify_high,option_b_grab_only,confidence
0,RC101,All,Watermelon,K2,Kill,stop,-671.0,-1151.0,-191.0,671.0,NaN,NaN,NaN,NaN,medium
1,RC101,All,Pineapple,K3,Modify + test,"quiet hours only (6-10, 22-5)",-611.0,-1276.0,54.0,0.0,656.0,-405.0,1717.0,NaN,low
2,RC101,All,Guava,K3,Modify + test,shallower discount (10%),-114.0,-548.0,349.0,0.0,126.0,-131.0,410.0,NaN,low
3,RC101,All,PassionFruit,K2,Kill,stop,-478.0,-824.0,-147.0,478.0,NaN,NaN,NaN,NaN,medium
4,RC101,All,MixedBerryPremium,K2,Kill,stop the berry bundle,-1411.0,-2205.0,-687.0,1411.0,NaN,NaN,NaN,NaN,high
5,RC102,Grab,All,K3,Modify + test,shallower discount (12%) + remove berry,-1865.0,-4415.0,267.0,0.0,311.0,-2380.0,2618.0,-1865.0,low
6,RC102,LINE MAN,All,K2 (Modify turns it positive),Modify + test,shallower discount (12%) + remove berry,-3335.0,-5075.0,-1605.0,0.0,784.0,-522.0,2171.0,0.0,low
7,RC103,Grab,All,K2,Kill,stop,-2980.0,-5261.0,-681.0,2980.0,NaN,NaN,NaN,-2980.0,high
8,RC103,LINE MAN,All,K3 (no option above 0),Kill,stop,-1426.0,-3273.0,355.0,1426.0,NaN,NaN,NaN,0.0,high


(e) RC103 on 25 of 52 high waste-risk kitchen-days: -2,502 THB a month
At lesson-6 prices promos make -12,891 THB a month. Kill rows save 6,965. Modify rows, if tests confirm: +7,803 (best options sum 1,877, low -3,438, high 6,917)
RC102 both platforms: -5,200 THB a month today, best option 1,095
RC103 both platforms: -4,405 THB a month today, best option -937


**Kill 5 rows (RC101 Watermelon, PassionFruit and berry; RC103 on both platforms): +7.0k THB a month, the fairly safe part.** Modify + test 4 rows (RC101 Pineapple and Guava, RC102 on both platforms): their best options sum to +1.9k a month (range -3.4k to +6.9k), optimistic until tested.

At lesson-6 prices promos lose 12.9k a month; after the decisions about +1.9k if the tests confirm, 0 if they all fail and stop. RC102 LINE MAN: stopping it (0) is the safe alternative to its +0.8k test. RC103 on high waste-risk kitchen-days still loses 2.5k a month.

## Findings -> next steps
1. Promos lose 14.6k THB CM a month after waste and side effects; 0 of 25 RC102/RC103 days and 7 of 31 RC101 item-days are positive -> stop RC101 on Watermelon, PassionFruit and berry, and RC103 on both platforms (+7.0k a month).
2. RC102 lifts +75% because of the double date, not the price -> test 12% instead of 20% without berry (+1.1k a month on both platforms; stopping on LINE MAN is the safe fallback); confidence low, new users not measured.
3. The best RC101 day (Guava, 16 Apr, +19k THB) was Songkran, not the promo: old items sold 63-70% above baseline that day -> test Pineapple (quiet hours) and Guava (10%) for 4 weeks, +0.8k a month at best.
4. Promo days waste more, not less (RC102: 2,045 vs 1,131 cups) -> prep promo days from the baseline, not from hoped-for demand (lesson 10).
5. Kill saves 7.0k a month; Modify adds about +1.9k only if tests confirm (range -3.4k to +6.9k) -> plan on the Kill savings and treat tests as upside; berry savings sit inside a berry delist.